# 105장 · RAG 화면의 오류 상태와 안내 문구

근거를 찾지 못했다는 정상 응답과 서버 응답을 끝까지 받지 못한 상태는 다릅니다. 각 상태에서 사용자에게 확실히 말할 수 있는 내용만 표시합니다.

근거 부족, 접근 거부, 저장 여부 미확인의 다음 행동을 각각 설계합니다. 재전송으로 중복 작업이 생길 수 있는 상황을 일반적인 조회 실패와 구분합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 근거 없음 상태의 결과 카드

Q03의 문안 예시　“현재 허용된 자료로 교체 일정을 확인할 수 없습니다. 담당자에게 근거를 확인하세요.” 처리 자체는 끝났으므로 진행 표시를 끄고 출처를 비웁니다. 다른 사람의 대화를 찾을 수 없는 경우에는 대화가 실제로 존재하는지 드러내지 않고 현재 선택한 대화와 접근 범위를 확인하도록 안내합니다.


### 셀 01 · 근거 없는 결과의 상태 카드

로컬 Q03의 업무 결과를 읽고 다음에 할 행동을 붙여 보세요.


In [1]:
result = {'status': 'no_evidence',
 'answer': '주어진 문서에서 확인할 수 없습니다. 담당자에게 확인하세요.',
 'sources': [],
 'mode': 'local_rule_fixture',
 'model_calls': 0}
if result["status"] == "no_evidence":
    print(result["answer"])
    print("다음 행동 · 담당자에게 근거 확인")

주어진 문서에서 확인할 수 없습니다. 담당자에게 확인하세요.
다음 행동 · 담당자에게 근거 확인


사용자가 규정에서 찾을 수 없는 일정을 어디서 확인할지 알게 됩니다. 검색이 실패했거나 서버가 응답하지 않은 상황과는 안내가 다릅니다.

근거가 없을 때는 담당자에게 내용을 확인하도록 안내합니다. 저장에 실패한 경우에는 답이 계산됐을 수도 있으므로 무조건 다시 보내라고 안내하면 중복 처리가 생길 수 있습니다. 같은 오류 색상을 쓰더라도 다음 행동은 실패 단계에 맞춰 달라져야 합니다.


#### Q03을 종료하고 담당자 확인을 안내하는 화면

```mermaid
flowchart TD
nresult["결과 수신"]
  nstate["상태 확인"]
  nmessage["안내 문구"]
  naction["다음 행동"]
  nresult --> nstate
  nstate --> nmessage
  nmessage --> naction
```

일정 근거를 찾지 못한 Q03은 완료된 질의입니다. 상태를 읽어 진행 표시를 끝내고 다음 행동을 안내합니다.

## 응답 단절과 저장 실패의 구분

권한 오류 문안 예시　“작업을 수행할 권한을 확인하지 못했습니다. 담당자에게 접근 범위를 확인하세요.” 전송 중단 문안 예시　“응답 전달 여부를 확인하지 못했습니다. 저장됐을 수 있으므로 같은 요청 기록을 먼저 확인하세요.” 실제 저장 성공·실패를 모른다면 완료 이력으로 옮기지 않습니다.


### 과제 02 · 선택 연습 · 저장 오류 안내

답은 계산했지만 저장이 실패한 save_failure의 상태를 보고, 다음 행동이 들어간 한 줄을 적으세요.

이 사례의 요청 ID는 save_failure이고 저장 상태는 failed입니다. 오류 코드 storage_failure는 저장 단계의 실패를 뜻합니다. 서버에 요청이 도착했는지조차 모르는 경우와 구분해 화면 문구를 정하세요.


In [2]:
storage_status = "failed"
# 저장 상태에 맞춰 다음 행동을 적으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>storage_status = &quot;failed&quot;
if storage_status == &quot;failed&quot;:
    print(&quot;계산한 답을 저장하지 못했습니다. 재전송 전에 저장 기록을 확인하세요.&quot;)</code></pre>
<p>실행 결과</p><pre><code>계산한 답을 저장하지 못했습니다. 재전송 전에 저장 기록을 확인하세요.</code></pre>
<p>실패한 대상은 저장입니다. 원문 근거가 없다는 문구를 보여 주거나 자동으로 재전송하면 문제의 위치를 숨기게 됩니다.</p>
</details>


#### 저장 실패를 안내하는 문장

<mark>근거 없음과 저장 실패에서는 사용자가 할 다음 행동이 다릅니다.</mark> 문서를 찾지 못하면 추가 자료를 확인할 수 있습니다. 이미 계산한 답의 저장이 실패했다면 같은 요청을 다시 보내기 전에 저장 기록부터 확인하게 안내하세요.

선수 개념 · 69장 FastAPI StreamingResponse와 줄 단위 JSON

## 사용자가 다시 눌러도 되는지 결정하는 오류 문안

104장에서 어떤 요청의 화면인지 정했습니다. 이제 검색 완료, 저장 실패, 응답 단절의 세 상황에서 사용자가 같은 버튼을 눌러도 되는지 판단할 문구를 작성합니다.

### 동일한 질문 뒤에 나타날 수 있는 세 안내

<table><thead><tr><th scope="col">관찰할 수 있는 사실</th><th scope="col">카드에 표시할 문장</th><th scope="col">다음 조작</th></tr></thead><tbody><tr><td>상태 no_evidence, 출처 0건</td><td>허용된 규정에서는 교체 일정을 확인할 수 없습니다.</td><td>담당자에게 일정 근거를 문의한다.</td></tr><tr><td>답 있음, storage_status=failed</td><td>계산한 답을 보여 드립니다. 이 답은 대화 기록에 저장되지 않았습니다.</td><td>요청 ID로 저장 상태를 확인한다.</td></tr><tr><td>연결 중단, 저장 결과 미수신</td><td>답변 전달이 중단됐습니다. 요청 처리 여부를 확인해 주세요.</td><td>같은 ID의 기록을 먼저 조회한다.</td></tr></tbody></table>

세 번째 상황에서는 서버가 이미 저장했을 수도 있습니다. 자동으로 같은 요청을 새 ID로 보내면 동일한 작업을 두 번 수행할 수 있습니다.

### 서로의 문안으로 행동 예측하기

동료에게 오류 코드와 내부 기록은 가리고 문구만 읽어 주세요. 동료가 “다시 검색한다”, “담당자에게 문의한다”, “같은 요청 기록을 확인한다” 중 어느 행동을 골랐는지 적습니다. 기대와 다르면 내부 오류 이름을 추가하기 전에 다음 행동 문장을 고칩니다.

<mark>전송이 끊긴 시점에는 저장 성공과 실패 어느 쪽도 단정할 수 없습니다.</mark> 서버가 저장을 마친 직후 연결이 끊겨도 화면에는 응답이 도착하지 않습니다. 접속 오류와 검색 결과 없음은 사용자에게 다른 안내가 필요합니다.

106장에서는 화면이 받은 요청 ID로 처리·저장·재조회 기록을 실제로 연결합니다.

## 돌아보기

106장에는 업무 상태와 오류 상태를 섞지 않는 화면 문구를 넘기세요. 아직 저장된 적이 있는지 모를 때는 자동으로 다시 보내지 않고 요청 ID로 조회하는 경로를 남깁니다.
